In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm

# 配置DeepSeek API
API_KEY = "sk-xxxxxxxxxxxxxxxxxxxxxx"  # 替换为您的实际API密钥
API_URL = "https://api.deepseek.com/chat/completions"

In [13]:
words=["米", "养猪", "养牛", "人参", "发电机", "竹木", "油菜籽", "马铃薯", "水利建设", "农机具"]

In [14]:
# 分类函数
def classify_word(word):
    """使用DeepSeek API对单个词语进行分类"""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    # 精心设计的分类提示词
    prompt = f"""
    你是一个专业的经济活动分类器。请将以下关于村落居民生产活动的词语分类：
    1. **农业**：直接涉及农作物种植、畜禽养殖、林产品采集等第一产业活动
    2. **工业**：涉及产品制造、加工、维修等第二产业活动
    3. **其他**：不属于以上两类的服务业或其他活动
    
    分类规则：
    - 农业类：粮食作物(如米、麦)、经济作物、畜禽养殖(养猪、养牛)、林产品(人参、药材)
    - 工业类：工厂生产(农具工厂)、设备维修(修配厂)、手工艺品制作
    - 其他：商业(小卖部)、旅游、教育、渔业(捕鱼)、运输等
    
    请只返回分类结果：农业、工业、其他
    
    需要分类的词语：{word}
    """
    
    payload = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,  # 低随机性确保分类稳定
        "max_tokens": 10
    }
    
    try:
        response = requests.post(API_URL, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()["choices"][0]["message"]["content"].strip()
        
        # 标准化输出结果
        if "农业" in result:
            return "农业"
        elif "工业" in result:
            return "工业"
        else:
            return "其他"
    
    except Exception as e:
        print(f"分类错误: {word} - {str(e)}")
        return "分类失败"

In [15]:
# 批量分类函数（带延时避免API限流）
def batch_classify(words_list, delay=0.5):
    results = []
    for word in words_list:
        category = classify_word(word)
        results.append((word, category))
        time.sleep(delay)  # 避免请求过快
    return results

In [16]:
# 执行分类
print("开始分类村落生产词语...")
classification_results = batch_classify(words)

# 创建结果DataFrame
df = pd.DataFrame(classification_results, columns=["词语", "类别"])

# 显示结果
print("\n分类结果:")
print(df)

# 统计各类别数量
category_counts = df['类别'].value_counts()
print("\n类别统计:")
print(category_counts)

# 保存结果到CSV（可选）
df.to_csv("村落生产活动分类.csv", index=False, encoding='utf-8-sig')
print("\n结果已保存到: 村落生产活动分类.csv")

开始分类村落生产词语...

分类结果:
     词语  类别
0     米  农业
1    养猪  农业
2    养牛  农业
3    人参  农业
4   发电机  工业
5    竹木  农业
6   油菜籽  农业
7   马铃薯  农业
8  水利建设  其他
9   农机具  工业

类别统计:
农业    7
工业    2
其他    1
Name: 类别, dtype: int64

结果已保存到: 村落生产活动分类.csv


In [ ]:
import requests

# 替换为你的实际 API 密钥
API_KEY = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxx"  

# API 验证函数
def verify_deepseek_api(api_key):
    url = "https://api.deepseek.com/models"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            print("✅ API 密钥有效！可用模型列表：")
            models = [model["id"] for model in response.json().get("data", [])]
            print("\n".join(models))
            return True
        else:
            print(f"❌ 验证失败 (HTTP {response.status_code}): {response.text}")
            return False
            
    except Exception as e:
        print(f"🚨 请求异常: {str(e)}")
        return False

# 执行验证
verify_deepseek_api(API_KEY)

✅ API 密钥有效！可用模型列表：
deepseek-chat
deepseek-reasoner


True

In [4]:
def classify_word(word):
    """使用 DeepSeek API 对单个词进行产业分类"""
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    # 分类提示词
    prompt = f"""
    你是一个专业的经济活动分类器。请将以下关于村落居民生产活动的词语分类：
    1. **农业**：直接涉及农作物种植、畜禽养殖、林产品采集等第一产业活动
    2. **工业**：涉及产品制造、加工、维修等第二产业活动
    3. **其他**：不属于以上两类的服务业或其他活动
    
    分类规则：
    - 农业类：粮食作物(如米、麦)、经济作物、畜禽养殖(养猪、养牛)、林产品(人参、药材)
    - 工业类：工厂生产(农具工厂)、设备维修(修配厂)、手工艺品制作
    - 其他：商业(小卖部)、旅游、教育、渔业(捕鱼)、运输等

    请只返回分类结果：农业、工业、其他
    需要分类的词语：{word}
    """

    payload = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 10
    }

    try:
        response = requests.post(API_URL, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()["choices"][0]["message"]["content"].strip()
        
        if "农业" in result:
            return "农业"
        elif "工业" in result:
            return "工业"
        else:
            return "其他"
    
    except Exception as e:
        print(f"分类错误: {word} - {str(e)}")
        return "分类失败"

def process_excel(file_path):
    """加载 Excel 文件并对分词后的内容进行分类"""
    # 加载 Excel 文件
    df = pd.read_excel(file_path)
    
    # 使用 tqdm 包裹分词列表，显示进度条
    tqdm.pandas(desc="Processing words")  # 设置进度条描述
    
    # 假设 'segmented_text' 是包含分词内容的列
    df['classification'] = df['segmented_text'].progress_apply(lambda x: classify_word(x))
    
    # 创建新的 DataFrame，包含三元组格式
    triples = pd.DataFrame({
        'COL_C': df['segmented_text'],
        'Element': df['segmented_text'],  # 假设每个词就是一个要素，如果有不同需求可以调整
        'Industry': df['classification']
    })
    
    # 输出为新的 Excel 文件
    output_path = file_path.replace('.xlsx', '_classified.xlsx')
    triples.to_excel(output_path, index=False)
    print(f"分类结果已保存到 {output_path}")

# 示例使用
file_path = r"E:\【论文】\heritage science\0602analysis\00people's commune_only_production.xlsx"  # 替换为你的文件路径
process_excel(file_path)

Processing words: 100%|██████████| 334/334 [29:40<00:00,  5.33s/it]

分类结果已保存到 E:\【论文】\heritage science\0602analysis\00people's commune_only_production_classified.xlsx
